In [1]:
# ── 라이브러리 임포트 ─────────────────────────────────────────────────────
import os
import sys
import json
import glob
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import deque

# sklearn 평가 지표: F1, Precision, Recall, Confusion Matrix, PR-Curve, Average Precision
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score,
)

# ── 경로 설정 ──────────────────────────────────────────────────────────────
# eval_f1.ipynb 는 services/inference/평가지표/ 에 위치.
# src/ · data/ · models/ 는 한 단계 위 services/inference/ 에 있음.
#   NB_DIR       = .../services/inference/평가지표
#   PROJECT_ROOT = .../services/inference      (src/data/models 의 부모)
NB_DIR       = os.path.abspath(".")
PROJECT_ROOT = os.path.abspath(os.path.join(NB_DIR, ".."))   # 🔧 상위 폴더로 이동
SRC_DIR      = os.path.join(PROJECT_ROOT, "src")

# 혹시 노트북을 src 바로 옆(같은 레벨)에 두고 실행하는 경우에도 동작하도록 fallback
if not os.path.isdir(SRC_DIR):
    SRC_DIR = os.path.join(NB_DIR, "src")
    PROJECT_ROOT = NB_DIR

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# 🔧 2026-04-21: Jupyter 커널이 구버전 모듈을 캐싱해 새 함수 import 가 실패하던 문제 방지.
#    importlib.reload 로 stale cache 를 강제 무효화 → 항상 최신 .py 를 반영.
for _mod_name in ("preprocessing", "inference_core"):
    if _mod_name in sys.modules:
        importlib.reload(sys.modules[_mod_name])

from preprocessing import step1_prepare_window_data
from inference_core import get_alarm_status_with_consecutive_delay  # 🔄 연속 지속 K 개 초과 게이트

# 연속 지속 알람 게이트 (inference_api.py 와 동일 — API 동작과 1:1 일치)
# 10분 집계 기준: K_caut/warn=5 → 50분 연속, K_crit=3 → 30분 연속
CONSECUTIVE_K_CAUTION  = 5
CONSECUTIVE_K_WARNING  = 5
CONSECUTIVE_K_CRITICAL = 3
_MAX_K = max(CONSECUTIVE_K_CAUTION, CONSECUTIVE_K_WARNING, CONSECUTIVE_K_CRITICAL)

# ── Matplotlib 한글 출력 설정 ─────────────────────────────────────────────
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "Malgun Gothic"   # Windows 한글 폰트
plt.rcParams["axes.unicode_minus"] = False      # 마이너스(-) 기호 깨짐 방지

print(f"NB_DIR      : {NB_DIR}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SRC_DIR     : {SRC_DIR}")


ImportError: cannot import name 'get_alarm_status_with_consecutive_delay' from 'inference_core' (/Users/jun/GitStudy/human_A/src/inference_core.py)

In [2]:
# ── 이진 분류 F1 + Confusion Matrix (Caution 기준) ─────────────────────────
# alarm_level ≥ 1 → Abnormal(1), 그 외 → Normal(0) 로 이진화한 뒤
# 도메인별로 2×2 혼동행렬과 Precision/Recall/F1 을 함께 출력한다.
from IPython.display import display

EVAL_LEVEL = 1  # Caution: alarm_level >= 1 → Abnormal

for dom in DOMAINS:
    df = domain_data[dom]["ts"]
    y_true = df["y_true"].values
    y_pred = (df["alarm_level"].values >= EVAL_LEVEL).astype(int)

    # ── 2×2 혼동행렬 원소 ────────────────────────────────────────────────
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tp = int(((y_pred == 1) & (y_true == 1)).sum())

    # ── 지표 (분모 0 방지) ───────────────────────────────────────────────
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0

    # ── 혼동행렬 형태의 표 (행=실제, 열=예측) ────────────────────────────
    cm = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual Normal (0)", "Actual Abnormal (1)"],
        columns=["Pred Normal (0)", "Pred Abnormal (1)"],
    )
    print(f"\n[{dom}]  Precision={p:.4f}  Recall={r:.4f}  F1={f1:.4f}")
    display(cm.style.background_gradient(cmap="Blues", axis=None).format("{:,}"))


NameError: name 'DOMAINS' is not defined

In [ ]:
# ── SHAP beeswarm — 도메인별 피처 기여도 ──────────────────────────────────
# 최초 1회만: pip install shap
import shap
import tensorflow as tf

N_BACKGROUND = 100    # KernelExplainer/GradientExplainer 의 baseline 샘플 수
N_EXPLAIN    = 500    # beeswarm 점 하나당 1 샘플

rng = np.random.default_rng(42)

for dom in DOMAINS:
    if dom not in MODELS_DATA:
        print(f"⚠ {dom}: 모델 누락 — 스킵")
        continue

    data   = MODELS_DATA[dom]
    model  = data["model"]
    scaler = data["scaler"]
    cfg    = data["config"]
    features = cfg["features"]

    # 학습/추론과 동일한 입력 구성 (누락 피처는 0)
    X = pd.DataFrame(index=df_agg.index, columns=features, dtype=float)
    for f in features:
        X[f] = df_agg[f].astype(float).values if f in df_agg.columns else 0.0
    X_scaled = scaler.transform(X.values).astype("float32")

    n = X_scaled.shape[0]
    bg_idx  = rng.choice(n, size=min(N_BACKGROUND, n), replace=False)
    exp_idx = rng.choice(n, size=min(N_EXPLAIN, n), replace=False)
    background = X_scaled[bg_idx]
    explain    = X_scaled[exp_idx]

    # AE 출력을 MSE 스칼라로 래핑 — SHAP 이 "이상 점수에 대한 기여" 를 계산하게 함
    inp = tf.keras.Input(shape=(X_scaled.shape[1],), dtype="float32")
    recon = model(inp, training=False)
    mse_out = tf.keras.layers.Lambda(
        lambda xs: tf.reduce_mean(tf.square(xs[0] - xs[1]), axis=1, keepdims=True),
        output_shape=(1,),
        name=f"{dom}_recon_mse",
    )([inp, recon])
    mse_model = tf.keras.Model(inp, mse_out)

    # 1차: GradientExplainer (빠름). 실패 시 KernelExplainer fallback.
    try:
        explainer = shap.GradientExplainer(mse_model, background)
        sv = explainer.shap_values(explain)
        if isinstance(sv, list):
            sv = sv[0]
        sv = np.asarray(sv)
        if sv.ndim == 3 and sv.shape[-1] == 1:
            sv = np.squeeze(sv, axis=-1)   # (N, F, 1) → (N, F)
        method = "GradientExplainer"
    except Exception as e:
        print(f"  ↳ [{dom}] GradientExplainer 실패 ({type(e).__name__}) — KernelExplainer 로 fallback")

        def f_mse(x):
            x = np.asarray(x, dtype=np.float32)
            r = model.predict(x, verbose=0)
            return np.mean((x - r) ** 2, axis=1)

        k_explain = explain[:100]          # KernelExplainer 는 N*nsamples 배 느림
        explainer = shap.KernelExplainer(f_mse, background)
        sv = explainer.shap_values(k_explain, nsamples=100)
        sv = np.asarray(sv)
        explain = k_explain
        method = "KernelExplainer (fallback, n=100)"

    # beeswarm summary plot — 점=샘플, 색=피처 값
    print(f"\n[{dom.upper()}] SHAP ({method}) — features={len(features)}, samples={len(explain)}")
    shap.summary_plot(
        sv,
        explain,
        feature_names=features,
        show=False,
        plot_size=(8, max(3, len(features) * 0.4)),
    )
    plt.suptitle(f"[{dom.upper()}] SHAP — impact on reconstruction MSE", y=1.02, fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── PPT Part 1: EDA 단계 패턴 + SHAP Top 5 (4 도메인 통합) ─────────────
# 좌: T1(학습 평균) ~ T6(평가 5분할) 의 flow_rate 추이
# 우: 4개 도메인 SHAP 평균 |SHAP| → 도메인 내 max 정규화 → 합산 Top 5
import shap
import tensorflow as tf

# ── (1) 좌측 EDA: flow_rate_l_min 단계별 평균 ─────────────────────────────
FLOW_COL = "flow_rate_l_min"
assert FLOW_COL in df_agg.columns, f"{FLOW_COL} 이 df_agg 에 없음"

mask_train_agg = (df_agg.index >= pd.Timestamp(TRAIN_RANGE[0])) & (df_agg.index <= pd.Timestamp(TRAIN_RANGE[1]))
mask_eval_agg  = (df_agg.index >= pd.Timestamp(EVAL_RANGE[0]))  & (df_agg.index <= pd.Timestamp(EVAL_RANGE[1]))

train_mean_flow = float(df_agg.loc[mask_train_agg, FLOW_COL].mean())
eval_slice      = df_agg.loc[mask_eval_agg, [FLOW_COL]].copy()
eval_slice["bucket"] = pd.qcut(np.arange(len(eval_slice)), 5, labels=["T2","T3","T4","T5","T6"])
eval_means = eval_slice.groupby("bucket", observed=True)[FLOW_COL].mean().reindex(["T2","T3","T4","T5","T6"])

X_AXIS   = ["T1","T2","T3","T4","T5","T6"]
NORMAL   = [train_mean_flow] * 6
CLOGGING = [train_mean_flow] + eval_means.tolist()

# ── (2) 우측 SHAP: 4 도메인 통합 Top 5 ─────────────────────────────────────
rng   = np.random.default_rng(42)
N_BG  = 60
N_EXP = 150

feat_score = {}
for dom, data in MODELS_DATA.items():
    model, scaler = data["model"], data["scaler"]
    features = data["config"]["features"]

    X = pd.DataFrame(index=df_agg.index, columns=features, dtype=float)
    for f in features:
        X[f] = df_agg[f].astype(float).values if f in df_agg.columns else 0.0
    X_scaled = scaler.transform(X.values).astype("float32")

    n   = X_scaled.shape[0]
    bg  = X_scaled[rng.choice(n, size=min(N_BG, n), replace=False)]
    exp = X_scaled[rng.choice(n, size=min(N_EXP, n), replace=False)]

    inp     = tf.keras.Input(shape=(X_scaled.shape[1],), dtype="float32")
    recon   = model(inp, training=False)
    mse_out = tf.keras.layers.Lambda(
        lambda xs: tf.reduce_mean(tf.square(xs[0] - xs[1]), axis=1, keepdims=True),
        output_shape=(1,),
    )([inp, recon])
    mse_model = tf.keras.Model(inp, mse_out)

    try:
        sv = shap.GradientExplainer(mse_model, bg).shap_values(exp)
        if isinstance(sv, list): sv = sv[0]
        sv = np.asarray(sv)
        if sv.ndim == 3 and sv.shape[-1] == 1:
            sv = np.squeeze(sv, axis=-1)
    except Exception:
        def _f_mse(x):
            r = model.predict(np.asarray(x, dtype=np.float32), verbose=0)
            return np.mean((x - r) ** 2, axis=1)
        sv = np.asarray(shap.KernelExplainer(_f_mse, bg).shap_values(exp[:60], nsamples=60))

    mean_abs = np.abs(sv).mean(axis=0)
    denom    = max(float(mean_abs.max()), 1e-12)
    for f, v in zip(features, mean_abs / denom):
        feat_score[f] = feat_score.get(f, 0.0) + float(v)

top5_s = sorted(feat_score.items(), key=lambda kv: -kv[1])[:5]
# barh 는 아래→위이므로 역순으로 뒤집어 그림
shap_names  = [k for k,_ in top5_s][::-1]
shap_values = [v for _,v in top5_s][::-1]

# ── (3) 시각화 ────────────────────────────────────────────────────────────
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 4.3))

ax_l.plot(X_AXIS, NORMAL,   "o--", color="#888", lw=1.5, markersize=7, label="Normal Flow")
ax_l.plot(X_AXIS, CLOGGING, "o-",  color="#c92a2a", lw=2.5, markersize=8, label="Clogging Flow")
ax_l.set_title("📈 EDA: 정상 vs 이상 패턴 분석", fontsize=12, fontweight="bold")
ax_l.set_ylabel(FLOW_COL)
ax_l.legend(loc="lower left", fontsize=9)
ax_l.grid(True, alpha=0.3)

ax_r.barh(shap_names, shap_values, color="#2b8a3e", alpha=0.9)
ax_r.set_title("🌳 SHAP: 변수 기여도 분석 (4 도메인 통합 Top 5)", fontsize=12, fontweight="bold")
ax_r.set_xlabel("Normalized Mean |SHAP| (도메인 내 max=1 정규화 후 합산)")
ax_r.grid(True, axis="x", alpha=0.3)
for i, v in enumerate(shap_values):
    ax_r.text(v, i, f" {v:.3f}", va="center", fontsize=8)

plt.suptitle("EDA 및 SHAP 기반 핵심 인자 분석", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── (4) 2줄 요약 ──────────────────────────────────────────────────────────
drop_pct = (CLOGGING[-1] - train_mean_flow) / (train_mean_flow + 1e-12) * 100
top_raw  = [k for k,_ in top5_s]
print(f"막힘 발생 시 유량의 단계적 감소({drop_pct:+.1f}%)와 함께 배관 내 압력·전류의 비정상 변동 패턴이 뚜렷하게 관찰됐습니다.")
print(f"모델 결정에 있어 {top_raw[0]}·{top_raw[1]} 이 가장 높은 기여도를 보이며, 핵심 탐지 인자로 확정되었습니다.")

# 후속 셀 재사용용 전역 캐시
GLOBAL_SHAP_SCORE = dict(feat_score)


In [3]:
# ── PPT Part 2: Global Feature Importance + 도메인별 탐지 F1 ──────────
# 좌: 피처별 재구성 오차 평균(eval 구간) → 도메인 내 정규화 → 합산 Top 5
# 우: 4 도메인이 "무엇을 감지했는가" — 탐지 시나리오명 + F1

# ── (1) Global Feature Importance ─────────────────────────────────────────
mask_eval_agg = (df_agg.index >= pd.Timestamp(EVAL_RANGE[0])) & (df_agg.index <= pd.Timestamp(EVAL_RANGE[1]))

global_score = {}
for dom, data in MODELS_DATA.items():
    model, scaler = data["model"], data["scaler"]
    features = data["config"]["features"]

    X = pd.DataFrame(index=df_agg.index, columns=features, dtype=float)
    for f in features:
        X[f] = df_agg[f].astype(float).values if f in df_agg.columns else 0.0
    X_scaled = scaler.transform(X.values).astype("float32")
    X_eval   = X_scaled[mask_eval_agg]

    recon        = model.predict(X_eval, batch_size=512, verbose=0)
    per_feat_err = np.mean((X_eval - recon) ** 2, axis=0)     # (F,)

    denom = max(float(per_feat_err.max()), 1e-12)
    for f, v in zip(features, per_feat_err / denom):
        global_score[f] = global_score.get(f, 0.0) + float(v)

top5_g   = sorted(global_score.items(), key=lambda kv: -kv[1])[:5]
g_names  = [k for k,_ in top5_g][::-1]
g_values = [v for _,v in top5_g][::-1]

# ── (2) 도메인별 대표 탐지 이름 + F1 ──────────────────────────────────────
DETECTION_NAMES = {
    "motor":     "모터 과전류·베어링 이상 탐지",
    "hydraulic": "배관 막힘·누수 탐지",
    "nutrient":  "양액 EC 제어 이상 탐지",
    "zone_drip": "점적핀 막힘 탐지",
}

det_rows = []
for dom, d in domain_data.items():
    sub    = d["ts"].loc[d["ts"]["mask_all"]]
    y_true = sub["y_true"].values
    y_pred = (sub["alarm_level"] >= 1).astype(int).values
    f1     = f1_score(y_true, y_pred, zero_division=0)
    det_rows.append((DETECTION_NAMES.get(dom, dom), f1, dom))

det_rows.sort(key=lambda r: r[1])
det_names = [r[0] for r in det_rows]
det_f1    = [r[1] for r in det_rows]

# ── (3) 시각화 ────────────────────────────────────────────────────────────
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 4.6))

ax_l.barh(g_names, g_values, color="#3b5bdb", alpha=0.9)
ax_l.set_title("🌳 전역적 변수 중요도 (Global Feature Importance)", fontsize=12, fontweight="bold")
ax_l.set_xlabel("Normalized Importance (4 도메인 합산)")
ax_l.grid(True, axis="x", alpha=0.3)
for i, v in enumerate(g_values):
    ax_l.text(v, i, f" {v:.3f}", va="center", fontsize=8)

ax_r.barh(det_names, det_f1, color="#2b8a3e", alpha=0.9)
ax_r.set_title("🔎 개별 이상 탐지 사례 분석 (도메인별 F1)", fontsize=12, fontweight="bold")
ax_r.set_xlabel("Detection F1 Score (Caution 기준, 4~5월 이상 구간)")
ax_r.set_xlim(0, 1.05)
ax_r.grid(True, axis="x", alpha=0.3)
for i, v in enumerate(det_f1):
    ax_r.text(v, i, f" {v:.3f}", va="center", fontsize=8)

plt.suptitle("XAI 기반 탐지 근거 시각화 (SHAP)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── (4) 요약 캡션 ─────────────────────────────────────────────────────────
top1_g = sorted(global_score.items(), key=lambda kv: -kv[1])[0]
best   = det_rows[-1]
worst  = det_rows[0]
print(f"모델 전체에서 {top1_g[0]} 이(가) 이상 탐지에 가장 결정적인 역할을 수행하며, 압력·유량·전류 계열 피처가 그 뒤를 잇습니다.")
print(f"도메인별로는 '{best[0]}' 모델이 F1 {best[1]:.3f} 로 가장 정확하게 시나리오를 분리해냈고,")
print(f"4개 도메인 모두 F1 {worst[1]:.3f} 이상을 유지 — 서로 다른 고장 모드를 독립 포착하는 상호보완 구조가 입증됐습니다.")


NameError: name 'df_agg' is not defined

In [ ]:
# ── PPT Part 1: EDA 단계 패턴 + SHAP Top 5 (4 도메인 통합) ─────────────
# 좌: T1(학습 평균) ~ T6(평가 5분할) 의 flow_rate 추이
# 우: 4개 도메인 SHAP 평균 |SHAP| → 도메인 내 max 정규화 → 합산 Top 5
import shap
import tensorflow as tf

# ── (1) 좌측 EDA: flow_rate_l_min 단계별 평균 ─────────────────────────────
FLOW_COL = "flow_rate_l_min"
assert FLOW_COL in df_agg.columns, f"{FLOW_COL} 이 df_agg 에 없음"

mask_train_agg = (df_agg.index >= pd.Timestamp(TRAIN_RANGE[0])) & (df_agg.index <= pd.Timestamp(TRAIN_RANGE[1]))
mask_eval_agg  = (df_agg.index >= pd.Timestamp(EVAL_RANGE[0]))  & (df_agg.index <= pd.Timestamp(EVAL_RANGE[1]))

train_mean_flow = float(df_agg.loc[mask_train_agg, FLOW_COL].mean())
eval_slice      = df_agg.loc[mask_eval_agg, [FLOW_COL]].copy()
eval_slice["bucket"] = pd.qcut(np.arange(len(eval_slice)), 5, labels=["T2","T3","T4","T5","T6"])
eval_means = eval_slice.groupby("bucket", observed=True)[FLOW_COL].mean().reindex(["T2","T3","T4","T5","T6"])

X_AXIS   = ["T1","T2","T3","T4","T5","T6"]
NORMAL   = [train_mean_flow] * 6
CLOGGING = [train_mean_flow] + eval_means.tolist()

# ── (2) 우측 SHAP: 4 도메인 통합 Top 5 ─────────────────────────────────────
rng   = np.random.default_rng(42)
N_BG  = 60
N_EXP = 150

feat_score = {}
for dom, data in MODELS_DATA.items():
    model, scaler = data["model"], data["scaler"]
    features = data["config"]["features"]

    X = pd.DataFrame(index=df_agg.index, columns=features, dtype=float)
    for f in features:
        X[f] = df_agg[f].astype(float).values if f in df_agg.columns else 0.0
    X_scaled = scaler.transform(X.values).astype("float32")

    n   = X_scaled.shape[0]
    bg  = X_scaled[rng.choice(n, size=min(N_BG, n), replace=False)]
    exp = X_scaled[rng.choice(n, size=min(N_EXP, n), replace=False)]

    inp     = tf.keras.Input(shape=(X_scaled.shape[1],), dtype="float32")
    recon   = model(inp, training=False)
    mse_out = tf.keras.layers.Lambda(
        lambda xs: tf.reduce_mean(tf.square(xs[0] - xs[1]), axis=1, keepdims=True),
        output_shape=(1,),
    )([inp, recon])
    mse_model = tf.keras.Model(inp, mse_out)

    try:
        sv = shap.GradientExplainer(mse_model, bg).shap_values(exp)
        if isinstance(sv, list): sv = sv[0]
        sv = np.asarray(sv)
        if sv.ndim == 3 and sv.shape[-1] == 1:
            sv = np.squeeze(sv, axis=-1)
    except Exception:
        def _f_mse(x):
            r = model.predict(np.asarray(x, dtype=np.float32), verbose=0)
            return np.mean((x - r) ** 2, axis=1)
        sv = np.asarray(shap.KernelExplainer(_f_mse, bg).shap_values(exp[:60], nsamples=60))

    mean_abs = np.abs(sv).mean(axis=0)
    denom    = max(float(mean_abs.max()), 1e-12)
    for f, v in zip(features, mean_abs / denom):
        feat_score[f] = feat_score.get(f, 0.0) + float(v)

top5_s = sorted(feat_score.items(), key=lambda kv: -kv[1])[:5]
# barh 는 아래→위이므로 역순으로 뒤집어 그림
shap_names  = [k for k,_ in top5_s][::-1]
shap_values = [v for _,v in top5_s][::-1]

# ── (3) 시각화 ────────────────────────────────────────────────────────────
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 4.3))

ax_l.plot(X_AXIS, NORMAL,   "o--", color="#888", lw=1.5, markersize=7, label="Normal Flow")
ax_l.plot(X_AXIS, CLOGGING, "o-",  color="#c92a2a", lw=2.5, markersize=8, label="Clogging Flow")
ax_l.set_title("📈 EDA: 정상 vs 이상 패턴 분석", fontsize=12, fontweight="bold")
ax_l.set_ylabel(FLOW_COL)
ax_l.legend(loc="lower left", fontsize=9)
ax_l.grid(True, alpha=0.3)

ax_r.barh(shap_names, shap_values, color="#2b8a3e", alpha=0.9)
ax_r.set_title("🌳 SHAP: 변수 기여도 분석 (4 도메인 통합 Top 5)", fontsize=12, fontweight="bold")
ax_r.set_xlabel("Normalized Mean |SHAP| (도메인 내 max=1 정규화 후 합산)")
ax_r.grid(True, axis="x", alpha=0.3)
for i, v in enumerate(shap_values):
    ax_r.text(v, i, f" {v:.3f}", va="center", fontsize=8)

plt.suptitle("EDA 및 SHAP 기반 핵심 인자 분석", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── (4) 2줄 요약 ──────────────────────────────────────────────────────────
drop_pct = (CLOGGING[-1] - train_mean_flow) / (train_mean_flow + 1e-12) * 100
top_raw  = [k for k,_ in top5_s]
print(f"막힘 발생 시 유량의 단계적 감소({drop_pct:+.1f}%)와 함께 배관 내 압력·전류의 비정상 변동 패턴이 뚜렷하게 관찰됐습니다.")
print(f"모델 결정에 있어 {top_raw[0]}·{top_raw[1]} 이 가장 높은 기여도를 보이며, 핵심 탐지 인자로 확정되었습니다.")

# 후속 셀 재사용용 전역 캐시
GLOBAL_SHAP_SCORE = dict(feat_score)
